# 02_clean — Clean corpus

> **⚑ Islamic Republic corpus (IRNA / Tasnim).** Isolated copy of the English notebook — reads `data/iran_raw/`, writes `data/interim/iran/` + `data/output/iran/`, and (where actor normalization is involved) uses `src/alias_map_iran`. The English pipeline and its outputs are untouched.


> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** `data/interim/corpus.jsonl`  
**Output:** `data/interim/corpus_clean.jsonl` and `data/output/figures/volume_curve.png`

## Pipeline steps in this notebook

1. Setup & paths
2. Load corpus.jsonl
3. Parse dates (string -> ISO date)
4. Assign time windows + drop out-of-window articles
5. Deduplicate wire-story republications
6. Remove stubs (body < 50 words)
7. Strip residual boilerplate footers
8. Re-assign sequential IDs
9. Corpus statistics
10. Daily volume curve
11. Write corpus_clean.jsonl

## Step 1: Setup & paths

In [ ]:
import re
import json
import sys
from pathlib import Path
from datetime import datetime, date
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.time_windows import assign_window

INTERIM_DIR = ROOT / 'data' / 'interim' / 'iran'
FIGURES_DIR = ROOT / 'data' / 'output' / 'iran' / 'analysis' / 'corpus'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

IN_FILE  = INTERIM_DIR / 'corpus.jsonl'
OUT_FILE = INTERIM_DIR / 'corpus_clean.jsonl'

print(f'Project root : {ROOT}')
print(f'Input        : {IN_FILE}')
print(f'Output       : {OUT_FILE}')
assert (ROOT / 'src').is_dir(), f'ERROR: src/ not found under {ROOT}'
assert IN_FILE.exists(),        f'ERROR: {IN_FILE} not found — run 01_parse first'

## Step 2: Load corpus.jsonl

In [ ]:
with open(IN_FILE, encoding='utf-8') as f:
    articles = [json.loads(line) for line in f]

n_raw = len(articles)
print(f'Loaded {n_raw} articles from corpus.jsonl')
assert n_raw > 0, 'corpus.jsonl is empty — re-run 01_parse'

## Step 3: Parse dates

Convert raw string dates (`'March 7, 2025'`) into Python `date` objects.
Six date format variants are tried because Nexis dates vary.

In [ ]:
DATE_FORMATS = [
    '%B %d, %Y',  # March 7, 2025
    '%B %d,%Y',
    '%b %d, %Y',  # Mar 7, 2025
    '%b %d,%Y',
]

def parse_date(raw: str | None) -> date | None:
    if not raw:
        return None
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(raw.strip(), fmt).date()
        except ValueError:
            continue
    return None

failed = []
for art in articles:
    art['_date_parsed'] = parse_date(art.get('date'))
    if art['_date_parsed'] is None:
        failed.append(art)

print(f'Date parse failures: {len(failed)} / {n_raw}')
if failed:
    print('Sample failures:')
    for a in failed[:5]:
        print(f"  id={a['id']}  raw_date={a.get('date')!r}")

## Step 4: Assign time windows + drop out-of-window articles

Windows are defined in `src/time_windows.py`. Articles outside Mar 1 – Aug 31 (or with invalid dates) are discarded here.

In [ ]:
in_window     = []
out_of_window = []
for art in articles:
    if art['_date_parsed'] is None:
        out_of_window.append(art)
        continue
    win = assign_window(art['_date_parsed'])
    if win is None:
        out_of_window.append(art)
    else:
        art['window'] = win
        in_window.append(art)

print(f'In-window articles  : {len(in_window)}')
print(f'Dropped (out/null)  : {len(out_of_window)}')
articles = in_window

## Step 5: Deduplicate wire-story republications

**Filter by duplicate** `(headline.strip().lower(), date)`
- Keeps the first occurrence.

In [ ]:
seen = set()
deduped = []
duplicates_dropped = []
for art in articles:
    key = ((art.get('headline') or '').strip().lower(), art['_date_parsed'])
    if key in seen:
        duplicates_dropped.append(art)
    else:
        seen.add(key)
        deduped.append(art)

print(f'After dedup    : {len(deduped)}  (dropped {len(duplicates_dropped)} wire-story duplicates)')
if duplicates_dropped:
    print('\nSample duplicates dropped:')
    for d in duplicates_dropped[:5]:
        print(f"  id={d['id']}  date={d['_date_parsed']}  source={d.get('source')!r}")
        print(f"    headline: {d.get('headline')!r}")
articles = deduped

## Step 6: Remove stubs (body < 50 words)

In [ ]:
STUB_THRESHOLD = 50
before = len(articles)
articles = [a for a in articles if len((a.get('body') or '').split()) >= STUB_THRESHOLD]
print(f'After stub removal : {len(articles)}  (dropped {before - len(articles)} stubs < {STUB_THRESHOLD} words)')

## Step 7: Strip residual boilerplate footers

01_parse already drops `Classification` and `Load-Date:` footers. This step is a belt-and-suspenders pass for any article whose body still contains a `Copyright 20XX` footer or unhandled tail. Footer markers are matched only at line starts (`\n`-anchored), so a mid-sentence mention such as *"…Phase Classification (IPC)…"* is never mistaken for the footer and the body is not truncated.

In [ ]:
BOILERPLATE_PAT = re.compile(r'\n(?:Load-Date:|End of Document|Classification|Copyright 20\d\d)')
for art in articles:
    body = art.get('body') or ''
    art['body'] = BOILERPLATE_PAT.split(body)[0].strip()
print('Boilerplate stripped.')

## Step 8: Re-assign sequential IDs

Re-assign article IDs after having removed some articles from corpus. 

In [ ]:
for i, art in enumerate(articles, start=1):
    art['id'] = f'{i:04d}'
print(f'IDs reassigned: 0001 .. {len(articles):04d}')

## Step 9: Corpus statistics

In [ ]:
by_source = Counter(a.get('source', 'UNKNOWN') for a in articles)
by_window = Counter(a['window'] for a in articles)

print(f'Total articles (raw)  : {n_raw}')
print(f'After all cleaning    : {len(articles)}')
print()
print('Per-source breakdown (top 20):')
for src, cnt in by_source.most_common(20):
    print(f'  {cnt:4d}  {src}')
print()
print('Per-window breakdown:')
for win, cnt in sorted(by_window.items()):
    print(f'  {win:20s}  {cnt}')

## Step 10: Daily volume curve

Plotted day-by-day across the full corpus window (Mar 1 – Aug 31).

In [ ]:
df = pd.DataFrame([{'date': a['_date_parsed']} for a in articles])
df['date'] = pd.to_datetime(df['date'])

# Daily index
full_range = pd.date_range(start='2025-03-01', end='2025-08-31', freq='D')
daily = df.set_index('date').resample('D').size().reindex(full_range, fill_value=0)

fig, ax = plt.subplots(figsize=(20, 4.6))
ax.bar(daily.index, daily.values, width=0.9, color='steelblue', alpha=0.85)
ax.set_title('Article volume — daily', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Articles per day')

# X-Axis titles every 14 days, minor every day
ax.xaxis.set_major_locator(mdates.DayLocator(interval=14))
ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.tick_params(axis='x', which='minor', length=2)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

# Room for annotations
y_top = ax.get_ylim()[1]
ax.set_ylim(0, y_top * 1.62)   # headroom for the stacked Jun 22/22/23 labels

# Expected spikes
# Jun 22 carries TWO events (US strikes, then the Iranian barrage on Israel
# hours later) and Jun 23 a third, so labels are stacked vertically and the
# vertical line is drawn once per date. Colour groups by DATE: purple =
# Jun 22 (US strikes + the Iranian barrage on Israel hours later), green =
# Jun 23 (Al Udeid).
_seen_dates = set()
for label, x_str, color, y_mult in [
    ('Israeli strikes begin', '2025-06-13', 'tab:red',    1.05),
    ('Iran strikes Israel',   '2025-06-22', 'tab:purple', 1.18),
    ('Midnight Hammer',       '2025-06-22', 'tab:purple', 1.31),
    ('Iran strikes Al Udeid', '2025-06-23', 'tab:green',  1.44),
    ('Snapback / FM',         '2025-08-15', 'tab:orange', 1.05),
]:
    x = pd.Timestamp(x_str)
    if x_str not in _seen_dates:
        ax.axvline(x, color=color, linestyle='--', linewidth=1, alpha=0.7)
        _seen_dates.add(x_str)
    ax.text(x, y_top * y_mult, label,
            fontsize=9, color=color, ha='center', va='bottom',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                      edgecolor=color, alpha=0.85, linewidth=0.8))

plt.tight_layout()
out_path = FIGURES_DIR / 'volume_curve.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {out_path}')

### Step 10b: Interactive Plotly version

Needed: `pip install plotly`

In [ ]:
try:
    import plotly.graph_objects as go
    fig = go.Figure(go.Bar(x=daily.index, y=daily.values, marker_color='steelblue'))
    EVENTS = [
        ('Israeli strikes begin', '2025-06-13', 'red',    1.04),
        ('Iran strikes Israel',   '2025-06-22', 'purple', 1.14),
        ('Midnight Hammer',       '2025-06-22', 'purple', 1.24),
        ('Iran strikes Al Udeid', '2025-06-23', 'green',  1.34),
        ('Snapback / FM',         '2025-08-15', 'orange', 1.04),
    ]
    _seen_dates = set()
    for label, x_str, color, y_label in EVENTS:
        if x_str not in _seen_dates:
            fig.add_shape(
                type='line', xref='x', yref='paper',
                x0=x_str, x1=x_str, y0=0, y1=1,
                line=dict(color=color, dash='dash', width=1.5),
            )
            _seen_dates.add(x_str)
        fig.add_annotation(
            x=x_str, y=y_label, yref='paper',
            text=label, showarrow=False,
            font=dict(color=color, size=11),
            yanchor='bottom', xanchor='center',
        )
    fig.update_layout(
        title='Article volume — daily (interactive)',
        xaxis_title='Date', yaxis_title='Articles per day',
        height=460,
        margin=dict(t=150),
        xaxis=dict(rangeslider=dict(visible=True), type='date'),
    )
    html_out = FIGURES_DIR / 'volume_curve.html'
    fig.write_html(html_out)
    print(f'Saved interactive chart to {html_out}')
    fig.show()
except ImportError:
    print('plotly not installed — skip this cell or run: pip install plotly')

### Step 10c: Cleaning validation — raw vs cleaned volume

The curve above validates the *sampling*: spikes land on the known event dates.
It says nothing about the *cleaning*, which is what this section is titled after.
This cell makes the missing comparison by re-reading the pre-cleaning corpus and
overlaying the two daily distributions.

The question it answers: did removing ~15 % of articles change the temporal
shape? Deduplication in particular could plausibly hit high-volume days hardest,
since wire stories cluster on big news days, which would flatten exactly the
spikes the analysis depends on.

In [ ]:
# Re-read the PRE-cleaning corpus; `articles` has already been filtered by now.
import numpy as np

with open(IN_FILE, encoding='utf-8') as f:
    _raw = [json.loads(line) for line in f]

_full = pd.date_range(start='2025-03-01', end='2025-08-31', freq='D')
_rd = pd.to_datetime(pd.Series([a.get('date') for a in _raw]),
                     format='%B %d, %Y', errors='coerce')
_cd = pd.to_datetime(pd.Series([a['_date_parsed'] for a in articles]))
raw_daily   = _rd.value_counts().reindex(_full, fill_value=0).sort_index()
clean_daily = _cd.value_counts().reindex(_full, fill_value=0).sort_index()

r_pearson  = np.corrcoef(raw_daily.values, clean_daily.values)[0, 1]
r_spearman = pd.Series(raw_daily.values).corr(pd.Series(clean_daily.values),
                                              method='spearman')
_busy = raw_daily.nlargest(12).index
ret_busy  = clean_daily[_busy].sum() / raw_daily[_busy].sum()
ret_other = ((clean_daily.sum() - clean_daily[_busy].sum())
             / (raw_daily.sum()  - raw_daily[_busy].sum()))

print(f'in-range raw       : {raw_daily.sum()}')
print(f'after cleaning     : {clean_daily.sum()}  ({clean_daily.sum()/raw_daily.sum():.1%} retained)')
print(f'Pearson  r         : {r_pearson:.4f}')
print(f'Spearman r         : {r_spearman:.4f}')
print(f'retention, 12 busiest days : {ret_busy:.1%}')
print(f'retention, all other days  : {ret_other:.1%}')
print(f'peak day raw / clean       : {raw_daily.idxmax().date()} / {clean_daily.idxmax().date()}')

fig, ax = plt.subplots(figsize=(20, 4.6))
# Cleaned is a subset of raw, so the grey sliver above each blue bar is exactly
# what cleaning removed on that day.
ax.bar(raw_daily.index, raw_daily.values, width=0.9,
       color='0.78', label='Before cleaning')
ax.bar(clean_daily.index, clean_daily.values, width=0.9,
       color='steelblue', alpha=0.95, label='After cleaning')
ax.set_title('Article volume before and after cleaning', fontsize=13)
ax.set_xlabel('Date'); ax.set_ylabel('Articles per day')
ax.xaxis.set_major_locator(mdates.DayLocator(interval=14))
ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.tick_params(axis='x', which='minor', length=2)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
ax.legend(loc='upper left', frameon=True)
ax.text(0.995, 0.94,
        f'Pearson r = {r_pearson:.3f}   Spearman r = {r_spearman:.3f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                  edgecolor='0.7', alpha=0.9))
plt.tight_layout()
_out = FIGURES_DIR / 'volume_curve_raw_vs_clean.png'
plt.savefig(_out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {_out}')


## Step 11: Write corpus_clean.jsonl

In [ ]:
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    for art in articles:
        f.write(json.dumps({
            'id':       art['id'],
            'date':     art['_date_parsed'].isoformat(),
            'source':   art.get('source'),
            'headline': art.get('headline'),
            'body':     art.get('body'),
            'window':   art['window'],
        }, ensure_ascii=False) + '\n')

print(f'Wrote {len(articles)} records to {OUT_FILE}')